<a href="https://colab.research.google.com/github/abdulhadi19306v10-oss/Model-training/blob/main/model%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets accelerate evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [22]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

train_dataset = dataset["train"].shuffle(seed=42)          # full 25k
test_dataset = dataset["test"].shuffle(seed=42).select(range(2000))  # bigger eval set

In [23]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [24]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

In [26]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="bert-imdb-512",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch = 16
    num_train_epochs=3,
    fp16=True,
    logging_steps=200,
    push_to_hub=False,
)

In [27]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.979592,0.215212,0.927000
2,0.642860,0.236668,0.939000
3,0.236813,0.345267,0.942000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4689, training_loss=0.6948736446654779, metrics={'train_runtime': 2433.8605, 'train_samples_per_second': 30.815, 'train_steps_per_second': 1.927, 'total_flos': 1.9733329152e+16, 'train_loss': 0.6948736446654779, 'epoch': 3.0})

In [10]:
trainer.train(resume_from_checkpoint=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert-imdb-full-final/tokenizer_config.json',
 'distilbert-imdb-full-final/tokenizer.json')

In [29]:
trainer.save_model("bert-imdb-512-final")
tokenizer.save_pretrained("bert-imdb-512-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert-imdb-512-final/tokenizer_config.json',
 'bert-imdb-512-final/tokenizer.json')

In [30]:
full_test = dataset["test"].map(tokenize_fn, batched=True)
trainer.evaluate(eval_dataset=full_test)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.236813,0.349639,3,0.940920


{'eval_loss': 0.3496389389038086, 'eval_accuracy': 0.94092}

In [31]:
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names=["Negative", "Positive"]))

              precision    recall  f1-score   support

    Negative       0.94      0.95      0.94      1000
    Positive       0.95      0.94      0.94      1000

    accuracy                           0.94      2000
   macro avg       0.94      0.94      0.94      2000
weighted avg       0.94      0.94      0.94      2000

